Submit a cleaned, analysis-ready dataset as a CSV, plus a .py script documenting every transformation
made and why. Include before/after row counts, data type confirmations, date consistency flag counts, and
a geographic data quality summary noting any invalid coordinate records.

In [ ]:
import pandas as pd

Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate,
ContractStatusChangeDate)
- • Remove unnecessary or redundant columns
- • Handle missing values appropriately
- • Ensure numeric fields are properly typed
- • Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0,
negative Bedrooms or Bathrooms

Validate the logical order of date fields: ListingContractDate should precede PurchaseContractDate,
which should precede CloseDate. Create boolean flag columns to mark records that violate these rules:
- • listing_after_close_flag
- • purchase_after_close_flag
- • negative_timeline_flag

Geographic Data Checks
- • Flag records with missing coordinates (Latitude or Longitude is null)
- • Flag Latitude = 0 or Longitude = 0 (sentinel null values)
- • Flag Longitude > 0 errors (California coordinates should be negative)
- • Flag out-of-state or implausible coordinates

In [2]:
sold = pd.read_csv("data/sold_enriched.csv", low_memory=False)
listing = pd.read_csv("data/listing_enriched.csv", low_memory=False)

In [3]:
sold["CloseDate"] = pd.to_datetime(sold["CloseDate"])
sold["PurchaseContractDate"] = pd.to_datetime(sold["PurchaseContractDate"])
sold["ListingContractDate"] = pd.to_datetime(sold["ListingContractDate"])
sold["ContractStatusChangeDate"] = pd.to_datetime(sold["ContractStatusChangeDate"])

listing["CloseDate"] = pd.to_datetime(listing["CloseDate"])
listing["PurchaseContractDate"] = pd.to_datetime(listing["PurchaseContractDate"])
listing["ListingContractDate"] = pd.to_datetime(listing["ListingContractDate"])
listing["ContractStatusChangeDate"] = pd.to_datetime(listing["ContractStatusChangeDate"])

In [4]:
# Listing after Purchase, Purachase after Close Date, Listing after Close Date
# Listing after Close should be redudant, but just in case:
sold["listing_after_close_flag"] = ((sold["ListingContractDate"] > sold["CloseDate"]))
sold["purchase_after_close_flag"] = ((sold["PurchaseContractDate"] > sold["CloseDate"]))
sold["negative_timeline_flag"] = ((sold["ListingContractDate"] > sold["PurchaseContractDate"]) | (sold["PurchaseContractDate"] > sold["CloseDate"]) | (sold["ListingContractDate"] > sold["CloseDate"]))

# Less than 1% of data has negative timeline -> FILTER OUT!
print("Percent of Sold Data with negative timelines:", sold[sold["negative_timeline_flag"]][["ListingContractDate", "PurchaseContractDate", "CloseDate"]].shape[0] / sold.shape[0] * 100, "%")

listing["listing_after_close_flag"] = ((listing["ListingContractDate"] > listing["CloseDate"]))
listing["purchase_after_close_flag"] = ((listing["PurchaseContractDate"] > listing["CloseDate"]))
listing["negative_timeline_flag"] = ((listing["ListingContractDate"] > listing["PurchaseContractDate"]) | (listing["PurchaseContractDate"] > listing["CloseDate"]) | (listing["ListingContractDate"] > listing["CloseDate"]))

# Less than 1% of data has negative timeline -> FILTER OUT!
listing[listing["negative_timeline_flag"]][["ListingContractDate", "PurchaseContractDate", "CloseDate"]].shape[0] / listing.shape[0] * 100
print("Percent of Listing Data with negative timelines:", listing[listing["negative_timeline_flag"]][["ListingContractDate", "PurchaseContractDate", "CloseDate"]].shape[0] / listing.shape[0] * 100, "%")

print("Sold Rows (before):", sold.shape[0])
print("Listing Rows (before):", listing.shape[0])
sold = sold[~sold["negative_timeline_flag"] & ~sold["purchase_after_close_flag"] & ~sold["listing_after_close_flag"]]
listing = listing[~listing["negative_timeline_flag"] & ~listing["purchase_after_close_flag"] & ~listing["listing_after_close_flag"]]
print("Sold Rows (after):", sold.shape[0])
print("Listing Rows (after):", listing.shape[0])

print("Filtered out data with negative timelines!")

Percent of Sold Data with negative timelines: 0.12313367971722713 %
Percent of Listing Data with negative timelines: 0.09564598983893709 %
Sold Rows (before): 414184
Listing Rows (before): 566673
Sold Rows (after): 413674
Listing Rows (after): 566131
Filtered out data with negative timelines!


In [5]:
sold_corr = sold.select_dtypes(include='number').corr()
listing_corr = listing.select_dtypes(include='number').corr()

print("Returning columns with correlations with another variable greater than 0.5\n")
print("Sold:")
for col in sold_corr:
    if any(abs(sold_corr[col]).sort_values(ascending=False).iloc[1:] > 0.5):
        print(abs(sold_corr[col]).sort_values(ascending=False).iloc[:5])
        print()

Returning columns with correlations with another variable greater than 0.5

Sold:
ListingKey                 1.000000
ListingKeyNumeric          1.000000
rate_30yr_fixed            0.663935
DaysOnMarket               0.111341
BuyerAgencyCompensation    0.062263
Name: ListingKey, dtype: float64

Latitude                 1.000000
Longitude                0.611231
YearBuilt                0.075342
BathroomsTotalInteger    0.063521
Stories                  0.035978
Name: Latitude, dtype: float64

Longitude       1.000000
Latitude        0.611231
YearBuilt       0.095442
ListPrice       0.052915
GarageSpaces    0.034560
Name: Longitude, dtype: float64

ListPrice                1.000000
BathroomsTotalInteger    0.517105
BedroomsTotal            0.339823
OriginalListPrice        0.208180
ClosePrice               0.205712
Name: ListPrice, dtype: float64

ListingKey                 1.000000
ListingKeyNumeric          1.000000
rate_30yr_fixed            0.663935
DaysOnMarket               0.1113

In [6]:
all(sold["ListingKeyNumeric"] == sold["ListingKey"])
# DROP!!! ListingKeyNumeric
sold.drop("ListingKeyNumeric", axis = 1, inplace = True)
print("Dropped \"ListingKeyNumeric\"!")


Dropped "ListingKeyNumeric"!


In [7]:
only_two_values = sold[sold[["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]].isna().sum(axis = 1) == 1][["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]]
one_one_value = sold[sold[["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]].isna().sum(axis = 1) == 2][["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]]
acre = 43560
# smallest apartment let's say is 150 sqft
def sq_feet(acres, unknown):
    if acres == acres:
        return acres * acre
    if unknown == unknown:
        if unknown < 150:
            return unknown * acre
        else:
            return unknown
    return None

print("Using function to impute Missing \"LotSizeSquareFeet\" by converting \"LotSizeAcres\" or \"LotSizeArea\" if avaialable.")
print("NOTE: For \"LotSizeArea\", if the value was less than 150, it was assumed to be acres rather than square footage")

print("\nMissingness Before:")
print(sold[["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]].isna().sum())

sold["LotSizeSquareFeet"] = sold.apply(lambda row: sq_feet(row["LotSizeAcres"], row["LotSizeArea"]) if row["LotSizeSquareFeet"] != row["LotSizeSquareFeet"] else row["LotSizeSquareFeet"], axis=1)

print("\nImputed Sold LotSize!")

print("\nMissingness After:")
print(sold[["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]].isna().sum())

print("Before dropping the redudent columns, let's check for impossible values (acres < square feet)...\n")

print(sold[sold["LotSizeAcres"] > sold["LotSizeSquareFeet"]])

print("There are no LotSizeAcres values larger than LotSizeSquareFeet! We are good to drop!")

sold.drop(["LotSizeAcres", "LotSizeArea"], axis = 1, inplace = True)

print("\nDropped LotSizeAcres and LotSizeArea !")

Using function to impute Missing "LotSizeSquareFeet" by converting "LotSizeAcres" or "LotSizeArea" if avaialable.
NOTE: For "LotSizeArea", if the value was less than 150, it was assumed to be acres rather than square footage

Missingness Before:
LotSizeSquareFeet    32068
LotSizeAcres         32470
LotSizeArea          31939
dtype: int64

Imputed Sold LotSize!

Missingness After:
LotSizeSquareFeet    31910
LotSizeAcres         32470
LotSizeArea          31939
dtype: int64
Before dropping the redudent columns, let's check for impossible values (acres < square feet)...

Empty DataFrame
Columns: [BuyerAgentAOR, ListAgentAOR, Flooring, ViewYN, PoolPrivateYN, OriginalListPrice, ListingKey, CloseDate, ClosePrice, Latitude, Longitude, LivingArea, ListPrice, DaysOnMarket, ListOfficeName, BuyerOfficeName, CoListOfficeName, ListAgentFullName, AssociationFeeFrequency, MLSAreaMajor, CountyOrParish, AttachedGarageYN, ParkingTotal, PropertySubType, LotSizeAcres, BuyerOfficeAOR, YearBuilt, StreetNumb

In [8]:
print("\nListing:")
for col in listing_corr:
    if any(abs(listing_corr[col]).sort_values(ascending=False).iloc[1:] > 0.5):
        print(abs(listing_corr[col]).sort_values(ascending=False).iloc[:5])
        print()  


Listing:
ListingKey           1.000000
ListingKeyNumeric    1.000000
rate_30yr_fixed      0.695473
DaysOnMarket         0.136617
Latitude             0.064541
Name: ListingKey, dtype: float64

ListingKey           1.000000
ListingKeyNumeric    1.000000
rate_30yr_fixed      0.695473
DaysOnMarket         0.136617
Latitude             0.064541
Name: ListingKeyNumeric, dtype: float64

LotSizeAcres         1.000000
LotSizeSquareFeet    1.000000
LotSizeArea          0.009551
BedroomsTotal        0.004278
Latitude             0.002713
Name: LotSizeAcres, dtype: float64

LotSizeSquareFeet    1.000000
LotSizeAcres         1.000000
LotSizeArea          0.121674
Stories              0.006197
YearBuilt            0.005026
Name: LotSizeSquareFeet, dtype: float64

rate_30yr_fixed      1.000000
ListingKeyNumeric    0.695473
ListingKey           0.695473
DaysOnMarket         0.110678
Latitude             0.038383
Name: rate_30yr_fixed, dtype: float64



In [9]:
all(listing["ListingKeyNumeric"] == listing["ListingKey"])
# DROP!!! ListingKeyNumeric
listing.drop("ListingKeyNumeric", axis = 1, inplace = True)
print("Dropped \"ListingKeyNumeric\"!")

Dropped "ListingKeyNumeric"!


In [10]:
print("Using function to impute Missing \"LotSizeSquareFeet\" by converting \"LotSizeAcres\" or \"LotSizeArea\" if avaialable.")
print("NOTE: For \"LotSizeArea\", if the value was less than 150, it was assumed to be acres rather than square footage")

print("\nMissingness Before:")
print(listing[["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]].isna().sum())
listing["LotSizeSquareFeet"] = listing.apply(lambda row: sq_feet(row["LotSizeAcres"], row["LotSizeArea"]) if row["LotSizeSquareFeet"] != row["LotSizeSquareFeet"] else row["LotSizeSquareFeet"], axis=1)

print("\nImputed Listing LotSize!")

print("\nMissingness After:")
print(listing[["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]].isna().sum())

print("Before dropping the redudent columns, let's check for impossible values (acres < square feet)...\n")

print(listing[listing["LotSizeAcres"] > listing["LotSizeSquareFeet"]])

print("There are no LotSizeAcres values larger than LotSizeSquareFeet! We are good to drop!")

listing.drop(["LotSizeAcres", "LotSizeArea"], axis = 1, inplace = True)

print("\nDropped LotSizeAcres and LotSizeArea !")

Using function to impute Missing "LotSizeSquareFeet" by converting "LotSizeAcres" or "LotSizeArea" if avaialable.
NOTE: For "LotSizeArea", if the value was less than 150, it was assumed to be acres rather than square footage

Missingness Before:
LotSizeSquareFeet    45833
LotSizeAcres         46537
LotSizeArea          45654
dtype: int64

Imputed Listing LotSize!

Missingness After:
LotSizeSquareFeet    45610
LotSizeAcres         46537
LotSizeArea          45654
dtype: int64
Before dropping the redudent columns, let's check for impossible values (acres < square feet)...

Empty DataFrame
Columns: [OriginalListPrice, ListingKey, CloseDate, ClosePrice, Latitude, Longitude, LivingArea, ListPrice, DaysOnMarket, ListOfficeName, BuyerOfficeName, CoListOfficeName, ListAgentFullName, AssociationFeeFrequency, MLSAreaMajor, CountyOrParish, MlsStatus, AttachedGarageYN, ParkingTotal, PropertySubType, LotSizeAcres, BuyerOfficeAOR, YearBuilt, BuyerAgencyCompensationType, StreetNumberNumeric, Bathroom

In [49]:
#ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

# Both All invalid values equal to zero
sold[sold["ClosePrice"] <= 0]["ClosePrice"].value_counts()
sold[sold["LivingArea"] <= 0]["LivingArea"].value_counts()

print(f"Only {sold[sold["ClosePrice"] <= 0]["ClosePrice"].shape[0]} ClosePrice less than/equal to 0, and only {sold[sold["LivingArea"] <= 0]["LivingArea"].shape[0]} LivingArea less than/equal to zero -> DROP!")

print("Rows before dropping invalid ClosePrice:", sold.shape[0])

dropped = sold.drop(sold[sold["ClosePrice"] <= 0]["ClosePrice"].index)

print("Rows before dropping invalid LivingArea:", dropped.shape[0])

#dropped[dropped["ClosePrice"] <= 0]
dropped.drop(dropped[dropped["LivingArea"] <= 0]["LivingArea"].index, inplace = True)

print("Rows after dropping both invalid ClosePrice and LivingArea:", dropped.shape[0])
print("\nReturning number of rows with LivingArea or ClosePrice less than or equal to 0:", dropped[(dropped["ClosePrice"] <= 0) | (dropped["LivingArea"] <= 0)].shape[0])


print("\nNegative Bathroom Count:", dropped[dropped["BathroomsTotalInteger"] < 0].shape[0])
print("Negative Bedroom Count:", dropped[dropped["BedroomsTotal"] < 0].shape[0])
print("No need to drop anything!")

print("\nNegative DaysOnMarket Count:", dropped[dropped["DaysOnMarket"] < 0].shape[0], "-> DROP!")

dropped.drop(dropped[dropped["DaysOnMarket"] < 0]["DaysOnMarket"].index, inplace = True)

print("Rows after dropping both invalid DaysOnMarket:", dropped.shape[0])

filtered_sold = dropped.copy()

Only 1 ClosePrice less than/equal to 0, and only 149 LivingArea less than/equal to zero -> DROP!
Rows before dropping invalid ClosePrice: 413674
Rows before dropping invalid LivingArea: 413673
Rows after dropping both invalid ClosePrice and LivingArea: 413524

Returning number of rows with LivingArea or ClosePrice less than or equal to 0: 0

Negative Bathroom Count: 0
Negative Bedroom Count: 0
No need to drop anything!

Negative DaysOnMarket Count: 47 -> DROP!
Rows after dropping both invalid DaysOnMarket: 413477


In [55]:
#ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

# Both All invalid values equal to zero
listing[listing["ClosePrice"] <= 0]["ClosePrice"].value_counts()
listing[listing["LivingArea"] <= 0]["LivingArea"].value_counts()

print(f"{listing[listing["ClosePrice"] <= 0]["ClosePrice"].shape[0]} ClosePrice less than/equal to 0, and only {listing[listing["LivingArea"] <= 0]["LivingArea"].shape[0]} LivingArea less than/equal to zero -> DROP!")

print("Rows before dropping invalid ClosePrice:", listing.shape[0])

dropped = listing.drop(listing[listing["LivingArea"] <= 0]["LivingArea"].index)

print("Rows after dropping both invalid ClosePrice and LivingArea:", dropped.shape[0])
print("\nReturning number of rows with LivingArea or ClosePrice less than or equal to 0:", dropped[(dropped["ClosePrice"] <= 0) | (dropped["LivingArea"] <= 0)].shape[0])


print("\nNegative Bathroom Count:", dropped[dropped["BathroomsTotalInteger"] < 0].shape[0])
print("Negative Bedroom Count:", dropped[dropped["BedroomsTotal"] < 0].shape[0])
print("No need to drop anything!")

print("\nNegative DaysOnMarket Count:", dropped[dropped["DaysOnMarket"] < 0].shape[0], "-> DROP!")

dropped.drop(dropped[dropped["DaysOnMarket"] < 0]["DaysOnMarket"].index, inplace = True)

print("Rows after dropping both invalid DaysOnMarket:", dropped.shape[0])

filtered_listing = dropped.copy()

0 ClosePrice less than/equal to 0, and only 375 LivingArea less than/equal to zero -> DROP!
Rows before dropping invalid ClosePrice: 566131
Rows after dropping both invalid ClosePrice and LivingArea: 565756

Returning number of rows with LivingArea or ClosePrice less than or equal to 0: 0

Negative Bathroom Count: 0
Negative Bedroom Count: 0
No need to drop anything!

Negative DaysOnMarket Count: 29 -> DROP!
Rows after dropping both invalid DaysOnMarket: 565727


Geographic Data Checks
- • Flag records with missing coordinates (Latitude or Longitude is null)
- • Flag Latitude = 0 or Longitude = 0 (sentinel null values)
- • Flag Longitude > 0 errors (California coordinates should be negative)
- • Flag out-of-state or implausible coordinates

In [85]:
filtered_sold["invalid_coord_flag"] = (filtered_sold[["Latitude", "Longitude"]].isna().any(axis = 1)) | (filtered_sold[["Latitude", "Longitude"]] == 0).any(axis = 1) | (filtered_sold["Longitude"] > 0) | (filtered_sold["Longitude"] < -125) | (filtered_sold["Longitude"] > -113) | (filtered_sold["Latitude"] < 32) | (filtered_sold["Latitude"] > 42.5)
print("Sold Data Invalid Coordinate Count:", filtered_sold["invalid_coord_flag"].sum())

filtered_listing["invalid_coord_flag"] = (filtered_listing[["Latitude", "Longitude"]].isna().any(axis = 1)) | (filtered_listing[["Latitude", "Longitude"]] == 0).any(axis = 1) | (filtered_listing["Longitude"] > 0) | (filtered_listing["Longitude"] < -125) | (filtered_listing["Longitude"] > -113) | (filtered_listing["Latitude"] < 32) | (filtered_listing["Latitude"] > 42.5)
print("Listing Data Invalid Coordinate Count:", filtered_listing["invalid_coord_flag"].sum())

Sold Data Invalid Coordinate Count: 15957
Listing Data Invalid Coordinate Count: 80553


In [86]:
filtered_listing.to_csv("listing_prepared.csv", index=False)

filtered_sold.to_csv("sold_prepared.csv", index=False)

print("\nCSVs Created!")


CSVs Created!
